# 프로젝트 Notebook 03. 전처리, 통합과 EDA

목표: 지역·점포 데이터를 품질 검사하고 관측 단위를 맞춰 통합한 뒤 질문 중심 EDA를 수행한다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "data").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 실행하세요.")
print("저장소:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
locations = pd.read_csv(ROOT / "data/sample/location_features.csv")
stores = pd.read_csv(ROOT / "data/sample/stores.csv")

## 1. 원본 품질표

In [ ]:
def quality_profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_rate": df.isna().mean(),
        "unique_n": df.nunique(dropna=False),
    })
display(quality_profile(locations))
display(quality_profile(stores))

## 2. 점포 정제와 행정동 집계

개별 점포를 행정동별·대분류별 점포 수로 집계하여 지역 표와 관측 단위를 맞춘다.

In [ ]:
clean_stores = stores.drop_duplicates("상가업소번호").copy()
clean_stores["행정동명"] = clean_stores["행정동명"].str.strip()
clean_stores[["위도", "경도"]] = clean_stores[["위도", "경도"]].apply(pd.to_numeric, errors="coerce")
clean_stores = clean_stores.dropna(subset=["행정동명", "위도", "경도"])

counts = clean_stores.pivot_table(
    index="행정동명", columns="상권업종대분류명",
    values="상가업소번호", aggfunc="nunique", fill_value=0
).reset_index()
display(counts)

## 3. 검증 가능한 병합

validate로 1:1을 확인하고 indicator로 미매칭 지역을 찾는다.

In [ ]:
merged = locations.merge(
    counts, on="행정동명", how="left",
    validate="one_to_one", indicator=True
)
print(merged["_merge"].value_counts())
assert len(merged) == len(locations)
display(merged.loc[merged["_merge"] != "both"])

## 4. 질문 1: 유동인구와 카페 수는 함께 증가하는가?

In [ ]:
r = merged["유동인구"].corr(merged["카페수"])
plt.figure(figsize=(7, 4))
plt.scatter(merged["카페수"], merged["유동인구"])
for _, row in merged.iterrows():
    plt.annotate(row["행정동명"], (row["카페수"], row["유동인구"]), fontsize=8)
plt.xlabel("카페 수(개)"); plt.ylabel("유동인구 지수")
plt.title(f"카페 수와 유동인구: r={r:.2f}")
plt.show()

표본은 12개 지역뿐이고 합성 데이터이므로 상관을 실제 목포시 관계로 일반화할 수 없다.

## 5. 질문 2: 경쟁 대비 잠재고객

In [ ]:
merged["카페당20대인구"] = merged["20대인구"] / merged["카페수"].replace(0, np.nan)
result = merged[["행정동명", "20대인구", "카페수", "카페당20대인구"]].sort_values(
    "카페당20대인구", ascending=False
)
display(result)

## 6. 독립 연습

1. 분포·집단비교·관계 유형의 분석 질문을 하나씩 작성한다.
2. 각 질문에 표와 그래프로 답한다.
3. 제목, 단위, 표본 수와 데이터 범위를 표시한다.
4. 관찰·해석·한계를 각각 두 문장 작성한다.
5. inner join으로 바꾸었을 때 누락되는 지역을 확인한다.